In [3]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub

In [4]:
from langchain.agents import create_react_agent, AgentExecutor


In [7]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")



In [8]:
search_tool = TavilySearchResults(max_results=2)

In [9]:
result = search_tool.invoke("Give me the latest news on ai")
result

[{'url': 'https://blog.google/innovation-and-ai/technology/ai/google-ai-updates-july-2026',
  'content': 'July brought a wave of AI advances to support developers, simplify everyday life, and protect local communities. We gave developers faster, more efficient Gemini models for scaling production agents and advancing robotics. We also expanded our creative suite with next-generation music and video tools, and made Search and Gemini Spark even more connected to your go-to daily apps. Plus, we announced a new skilled trades alliance and new satellites for early wildfire detection — and Google DeepMind Co-Founder and CEO Demis Hassabis shared his vision for this critical moment in history. Together, our July updates showed how AI can be more practical, creative, and helpful than ever before. [...] Breadcrumb\n\n1. Home\n2. Innovation & AI\n3. Technology\n4. AI\n\n# The latest AI news we announced in July 2026\n\nAug 04, 2026\n\n|\n\n x.com\n Facebook\n LinkedIn\n Mail\n\nHere’s a recap of

In [10]:
#LLM

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key= GROQ_API_KEY
)

In [12]:
response=llm.invoke("which year is it?")
response

AIMessage(content="I'm not currently aware of the year. I'm a large language model, I don't have real-time access to the current date or year. However, I can tell you that my training data is cut off in 2023, so I may not have information on events or developments that have occurred after that.", response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 40, 'total_tokens': 105, 'completion_time': 0.174991727, 'completion_tokens_details': None, 'prompt_time': 0.000953019, 'prompt_tokens_details': None, 'queue_time': 0.162658519, 'total_time': 0.175944746}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'finish_reason': 'stop', 'logprobs': None}, id='run-881b490c-2597-4706-bf60-39655a521463-0')

In [13]:
#PROMPT

prompt = hub.pull("hwchase17/react")

d:\AI\AgenticAI\LangchainAgent\.venv\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [14]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [15]:
#TOOL

tools = [search_tool]

In [20]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [21]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://www.crescendo.ai/news/ai-breakthroughs-and-latest-ai-news-2026',
  'content': 'Date: January 29, 2026\n\nSummary: Amazon has announced a fresh wave of layoffs impacting approximately 16,000 corporate employees, citing a strategic shift toward AI-driven automation and "agentic" workflows. The job cuts primarily target middle management and administrative roles that have become redundant as the company integrates more sophisticated AI systems to handle logistics planning, vendor relations, and internal reporting. While Amazon continues to hire for AI-specialized engineering roles, the move reflects a broader industry trend of "re-skilling" and restructuring as tech giants prioritize high-efficiency AI infrastructure over traditional staffing models. The company stated these cuts are necessary to remain agile in a rapidly evolving, AI-first retail landscape. [...] Date: January 5, 2026\n\nSummary: Alibaba Group Holding Ltd. has launched a new AI-driven service through it

In [16]:
#CREATE AGENT

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [17]:
#EXECUTOR

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [24]:
#RUN

response = agent_executor.invoke({
    "input":(
        "find the financial capital of india"
        "and its current weather as well with the date and time mentioned"
    )
})





> Entering new AgentExecutor chain...


d:\AI\AgenticAI\LangchainAgent\.venv\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
d:\AI\AgenticAI\LangchainAgent\.venv\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
d:\AI\AgenticAI\LangchainAgent\.venv\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
d:\AI\Agentic

Thought: To find the financial capital of India and its current weather, I need to first identify the financial capital of India. I know that Mumbai is often referred to as the financial capital of India. Next, I need to find the current weather in Mumbai. I can use the tavily_search_results_json tool to search for the current weather in Mumbai.

Action: tavily_search_results_json
Action Input: "current weather in Mumbai"[{'url': 'https://www.indiatoday.in/weather/mumbai-weather-forecast-today', 'content': "## Weather In Mumbai\n\nThe minimum temperature in Mumbai today is likely to hover around 27 degrees Celsius, while the maximum temperature might reach 30 degrees Celsius. The mercury level is expected to hover around 29 degrees Celsius throughout the day, with the wind speed around 6.39. The wind will move around 256 degrees with a gust speed of 15.28. The sunrise time is 06:18 AM, while it will set at 07:07 PM on Thursday. As per the seven-day weather prediction, the temperature i

d:\AI\AgenticAI\LangchainAgent\.venv\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
d:\AI\AgenticAI\LangchainAgent\.venv\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
d:\AI\AgenticAI\LangchainAgent\.venv\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
d:\AI\Agentic

Thought: I have found the current weather in Mumbai, which is the financial capital of India. The current temperature is around 29 degrees Celsius, with a minimum temperature of 27 degrees Celsius and a maximum temperature of 30 degrees Celsius. The humidity is 82% and the wind speed is around 6.39 km/h. The weather forecast for the next few days shows a mix of clouds and rain. I also have information about the weather in Mumbai in August, including average temperatures, precipitation, and sunshine hours.

Action: None
Action Input: NoneNone is not a valid tool, try one of [tavily_search_results_json].

d:\AI\AgenticAI\LangchainAgent\.venv\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
d:\AI\AgenticAI\LangchainAgent\.venv\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
d:\AI\AgenticAI\LangchainAgent\.venv\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
d:\AI\Agentic

Thought: I now know the final answer
Final Answer: The financial capital of India is Mumbai, and its current weather as of August 13, 2026, is around 29 degrees Celsius, with a minimum temperature of 27 degrees Celsius and a maximum temperature of 30 degrees Celsius. The humidity is 82% and the wind speed is around 6.39 km/h. The weather forecast for the next few days shows a mix of clouds and rain. The current date and time is August 13, 2026, 10:24:37 AM IST.

> Finished chain.


d:\AI\AgenticAI\LangchainAgent\.venv\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
d:\AI\AgenticAI\LangchainAgent\.venv\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
d:\AI\AgenticAI\LangchainAgent\.venv\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
d:\AI\Agentic

In [25]:
print(response["output"])

The financial capital of India is Mumbai, and its current weather as of August 13, 2026, is around 29 degrees Celsius, with a minimum temperature of 27 degrees Celsius and a maximum temperature of 30 degrees Celsius. The humidity is 82% and the wind speed is around 6.39 km/h. The weather forecast for the next few days shows a mix of clouds and rain. The current date and time is August 13, 2026, 10:24:37 AM IST.
